# Oil and Gas Anomaly Detection — Section 4a: Experiment 3, Hyperparameter Search (Optuna)

This notebook contains the **hyperparameter optimisation** for Experiment 3. It uses the same data, split and noise as notebook `04b`. The best values it found are the fixed hyperparameters used in notebook `04b`.

| Setting | Value |
|---|---|
| Environment | Google Colab, GPU (T4); RAPIDS **cuML** / **CuPy** for `StandardScaler` and `SVC` |
| Input data | `final_datav1.csv` (Colab path `/content/final_datav1.csv`), `index` column dropped |
| Split | `train_test_split(test_size=0.2, random_state=42)` |
| Normalisation | cuML `StandardScaler` (GPU), fitted on the training set |
| Noise | `add_gaussian_noise(std=0.02)`, clipped to `[0, 1]`, applied to train and test sets (no seed) |
| Optimiser | Optuna, `create_study(direction='maximize')`, default sampler (TPE), no sampler seed |
| Trials | `n_trials=10` per model |
| Objective | **macro recall** of the model's predictions on the **test set** |
| Logged metrics | accuracy, macro recall, macro precision, macro F1, MCC (stored as trial user attributes) |
| Models searched | Lasso, Ridge, Elastic-Net logistic regression, SVC, k-NN, Decision Tree, Random Forest, Logistic Regression |

> **Saved outputs.** Only the Lasso, Ridge and Elastic-Net searches have saved outputs in the original notebook. The SVC, k-NN, Decision Tree, Random Forest and Logistic Regression search cells have **no saved outputs**, so their trial history is not recorded here. The notebook has **no search for XGBoost**.

*Documentation (markdown cells and `#` comment lines) was added for publication; the original executable code and its outputs are unchanged.*

## 0. Environment setup

In [ ]:
# ============================================================
# 0. ENVIRONMENT SETUP
# ============================================================
# Install Optuna in the Colab runtime (IPython shell command).
!pip install -q optuna

In [ ]:
# Import data-handling, modelling, tuning and metric libraries.
# Several imports (IsolationForest, AdaBoostClassifier, GaussianNB,
# OneClassSVM, LGBMClassifier, GridSearchCV, cross-validation utilities)
# are not used below; they are kept unchanged.
import pandas as pd
import numpy as np
import os
import glob

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import IsolationForest
from sklearn.ensemble import AdaBoostClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import OneClassSVM
from lightgbm import LGBMClassifier
#from catboost import CatBoostClassifier

from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold, cross_val_score

from sklearn.metrics import accuracy_score, recall_score, f1_score, precision_score, make_scorer, matthews_corrcoef

from sklearn.model_selection import StratifiedKFold, train_test_split, cross_val_score , GroupKFold, GroupShuffleSplit,StratifiedGroupKFold
import optuna
import warnings
from sklearn.exceptions import ConvergenceWarning
# Ignorer uniquement les ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
# GPU libraries (RAPIDS): this cuML StandardScaler replaces scikit-learn's,
# and CuPy is used to move arrays back to the CPU for the metrics.
from cuml.preprocessing import StandardScaler
import cupy as cp # GPU arrays

In [ ]:
# GPU support vector classifier from cuML. It replaces the scikit-learn SVC
# imported above, so the SVC search below runs on cuML.
from cuml.svm import SVC

## 1. Load the feature-engineered dataset

The same pre-assembled dataset as notebook `04b` (`final_datav1.csv`), loaded from the Colab working directory.

In [ ]:
# ============================================================
# 1. DATA LOADING
# ============================================================
# Load the pre-assembled feature dataset (128 features + `index` + `class`).
final_data = pd.read_csv('/content/final_datav1.csv')

In [ ]:
# Display the dataset.
final_data

,index,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,0,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
1,1,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
2,2,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
3,3,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
4,4,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117396,165104,1.715209e+07,1519.275871,1.714959e+07,1.715460e+07,1.715209e+07,5010.000000,1.715084e+07,1.715334e+07,2504.375000,...,4.365014,0.028755,0.014873,4.350611,4.350645,567.843367,-0.012862,-1.199806,0.0,9
117397,165105,1.715726e+07,1515.598171,1.715477e+07,1.715976e+07,1.715726e+07,4990.000000,1.715601e+07,1.715851e+07,2495.000000,...,4.304973,0.029790,0.015409,4.290051,4.290088,552.145641,-0.012840,-1.199792,0.0,9
117398,165106,1.716246e+07,1537.051439,1.715994e+07,1.716500e+07,1.716245e+07,5060.000000,1.716119e+07,1.716372e+07,2532.500000,...,4.242770,0.041188,0.020734,4.221550,4.221617,534.661449,-0.135992,-1.261894,0.0,9
117399,165107,1.716770e+07,1526.832665,1.716517e+07,1.717016e+07,1.716771e+07,4992.499999,1.716645e+07,1.716898e+07,2526.874999,...,4.155836,0.043605,0.022554,4.133993,4.134075,512.717264,-0.013416,-1.199885,0.0,9


In [ ]:
# Drop the stored row-index column.
final_data.drop(columns=['index'], inplace=True)

In [ ]:
# Display the first rows of the modelling dataset.
final_data.head()

,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,P-PDG_mad,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


# Function to add noise

### Noise injection (same function and parameters as Experiments 2 and 3)

`add_gaussian_noise(X, mean=0.0, std=0.02)` adds zero-mean Gaussian noise with standard deviation **0.02** to every feature value and then clips the result to the interval **[0, 1]** (`np.clip(X + noise, 0., 1.)`).

It is applied **after** standardisation, independently to the training set and to the test set. No random seed is set for `np.random`, so the noise realisation differs between runs.

In [ ]:
# ============================================================
# NOISE INJECTION FUNCTION
# ============================================================
# Add Gaussian noise N(mean, std^2) to every element of X, then clip the
# noisy values to [0, 1]. Defaults: mean=0.0, std=0.02. No seed is set.
def add_gaussian_noise(X, mean=0.0, std=0.02):
    noise = np.random.normal(loc=mean, scale=std, size=X.shape)
    return np.clip(X + noise, 0., 1.)

## 2. Train-test split, normalisation and noise injection

* Features `X` = the 128 statistical features; target `Y` = `class` (10 classes, 0–9).
* **Split:** `train_test_split(test_size=0.2, random_state=42)` — 80 % training / 20 % test, random (not stratified).
* **Normalisation:** cuML `StandardScaler` (GPU) fitted on the training set only, then applied to the test set.

In [ ]:
# ============================================================
# 2. TRAIN-TEST SPLIT, NORMALISATION AND NOISE INJECTION
# ============================================================
# Separate the 128 features (X) from the class label (Y).
X = final_data.drop('class', axis = 1)
Y = final_data['class']

In [ ]:
# Random 80/20 train-test split with a fixed seed (random_state=42), same as notebook 04b.
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

In [ ]:
# Standardise features with the cuML (GPU) scaler: fit on the training set, apply to both sets.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
# Inject Gaussian noise (std=0.02, clipped to [0, 1]) into the scaled training and test sets.
X_train = add_gaussian_noise(X_train, std=0.02)
X_test  = add_gaussian_noise(X_test, std=0.02)

## 3. Optuna hyperparameter search

Every model follows the same pattern:

1. an `objective(trial)` function samples hyperparameters from the search space below;
2. the model is trained on the noisy training set and predicts the noisy test set;
3. accuracy, macro recall, macro precision, macro F1 and MCC are computed with the standard argument order `metric(y_true, y_pred)` and logged on the trial;
4. the objective returns the **macro recall**, which Optuna maximises over **10 trials**;
5. the best trial's value, parameters and logged metrics are printed.

**Search spaces** (`log` = log-uniform sampling):

| Model | Searched hyperparameters | Fixed settings |
|---|---|---|
| Lasso (L1) LR | `C` ∈ [1e-4, 10] (log), `solver` ∈ {liblinear, saga}, `max_iter` ∈ [100, 1000] | `penalty='l1'`, `random_state=42` |
| Ridge (L2) LR | `C` ∈ [1e-4, 10] (log), `solver` ∈ {lbfgs, newton-cg, saga}, `max_iter` ∈ [100, 1000] | `penalty='l2'`, `random_state=42` |
| Elastic-Net LR | `C` ∈ [1e-4, 10] (log), `l1_ratio` ∈ [0, 1], `max_iter` ∈ [100, 1000] | `penalty='elasticnet'`, `solver='saga'`, `random_state=42` |
| SVC (cuML) | `C` ∈ [1e-3, 100] (log), `kernel` ∈ {linear, rbf, poly, sigmoid}, `gamma` ∈ {scale, auto}, `degree` ∈ [2, 5] | `probability=False`, `max_iter=1000` |
| k-NN | `n_neighbors` ∈ [3, 25], `weights` ∈ {uniform, distance}, `algorithm` ∈ {brute}, `metric` ∈ {euclidean, manhattan, minkowski}, `p` ∈ [1, 3] | — |
| Decision Tree | `criterion` ∈ {gini, entropy}, `max_depth` ∈ [3, 20], `min_samples_split` ∈ [2, 10], `min_samples_leaf` ∈ [1, 10] | — |
| Random Forest | `n_estimators` ∈ [100, 500], `max_depth` ∈ [4, 20], `min_samples_split` ∈ [2, 10], `min_samples_leaf` ∈ [1, 10], `bootstrap` ∈ {True, False} | — |
| Logistic Regression | `C` ∈ [1e-3, 100] (log), `penalty` ∈ {l2}, `fit_intercept` ∈ {True, False}, `max_iter` ∈ [100, 1000], `tol` ∈ [1e-6, 1e-3] (log) | — |

**Best trials recorded in the saved outputs:**

| Model | Best macro recall | Best parameters | Used in notebook `04b` |
|---|---|---|---|
| Lasso (L1) LR | 0.9307 | `C=2.3167281368521384, solver='saga', max_iter=935` | `C=3.167281368521384, solver='saga', max_iter=935` (**C differs**) |
| Ridge (L2) LR | 0.9266 | `C=7.249509828331678, solver='newton-cg', max_iter=786` | identical |
| Elastic-Net LR | 0.9017 | `C=0.32421631749317886, l1_ratio=0.26412068295144886, max_iter=797` | identical |

> The search uses the **test set** as the selection set (there is no separate validation set or cross-validation), so the tuned scores are optimistic estimates.

# Laaso

In [ ]:
# ============================================================
# 3.1 LASSO (L1) LOGISTIC REGRESSION - OPTUNA SEARCH
# ============================================================
# Search space: C in [1e-4, 10] (log), solver in {liblinear, saga},
# max_iter in [100, 1000]; fixed penalty='l1', random_state=42.
# Objective: train on the noisy training set, predict the noisy test set,
# log accuracy / macro recall / macro precision / macro F1 / MCC, and
# return the macro recall. Optuna maximises it over 10 trials.
def objective(trial):
    # Hyperparameters for Lasso Logistic Regression
    params = {
        'C': trial.suggest_float('C', 0.0001, 10.0, log=True),  # inverse of regularization strength
        'solver': trial.suggest_categorical('solver', ['liblinear', 'saga']),  # both support L1
        'max_iter': trial.suggest_int('max_iter', 100, 1000),
        'penalty': 'l1',  # LASSO regularization
        'random_state': 42
    }

    model = LogisticRegression(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    # Convert CuPy arrays to NumPy (if using GPU arrays)
    y_test_cpu = cp.asnumpy(y_test)
    preds_cpu = cp.asnumpy(preds)

    # Evaluation metrics
    f1 = f1_score(y_test_cpu, preds_cpu, average='macro')
    acc = accuracy_score(y_test_cpu, preds_cpu)
    rec = recall_score(y_test_cpu, preds_cpu, average='macro')
    preci = precision_score(y_test_cpu, preds_cpu, average='macro')
    matt = matthews_corrcoef(y_test_cpu, preds_cpu)

    # Log extra metrics
    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("recall", rec)
    trial.set_user_attr("precision", preci)
    trial.set_user_attr("matthews_corrcoef", matt)
    trial.set_user_attr("f1", f1)

    print(f"Trial {trial.number}: F1={f1:.4f}, Acc={acc:.4f}, Rec={rec:.4f}, Prec={preci:.4f}, MCC={matt:.4f}")

    return rec  # optimize recall

# Run the optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# Show the best results
best_trial = study.best_trial
print("\nBest REC:", best_trial.value)
print("Best Params:", best_trial.params)
print("Other metrics:", best_trial.user_attrs)


[I 2025-11-12 14:28:37,991] A new study created in memory with name: no-name-a6e5e9b2-82f2-41e9-aa69-d30c3bc6412f
[I 2025-11-12 14:42:29,185] Trial 0 finished with value: 0.9306994742509216 and parameters: {'C': 2.3167281368521384, 'solver': 'saga', 'max_iter': 935}. Best is trial 0 with value: 0.9306994742509216.


Trial 0: F1=0.9456, Acc=0.9701, Rec=0.9307, Prec=0.9666, MCC=0.9653


[I 2025-11-12 14:53:40,289] Trial 1 finished with value: 0.9302208553111019 and parameters: {'C': 1.8697504976821666, 'solver': 'saga', 'max_iter': 779}. Best is trial 0 with value: 0.9306994742509216.


Trial 1: F1=0.9452, Acc=0.9699, Rec=0.9302, Prec=0.9663, MCC=0.9650


[I 2025-11-12 14:54:09,522] Trial 2 finished with value: 0.8742720700324187 and parameters: {'C': 0.03763660829389721, 'solver': 'liblinear', 'max_iter': 807}. Best is trial 0 with value: 0.9306994742509216.


Trial 2: F1=0.8976, Acc=0.9466, Rec=0.8743, Prec=0.9442, MCC=0.9381


... (10 outputs left out)

[I 2025-11-12 15:16:27,927] Trial 8 finished with value: 0.91176599542655 and parameters: {'C': 0.2012191825465806, 'solver': 'saga', 'max_iter': 456}. Best is trial 0 with value: 0.9306994742509216.


Trial 8: F1=0.9297, Acc=0.9618, Rec=0.9118, Prec=0.9566, MCC=0.9557


[I 2025-11-12 15:19:02,749] Trial 9 finished with value: 0.8218028928216056 and parameters: {'C': 0.0061377371155461645, 'solver': 'saga', 'max_iter': 188}. Best is trial 0 with value: 0.9306994742509216.


Trial 9: F1=0.8420, Acc=0.9238, Rec=0.8218, Prec=0.9098, MCC=0.9117

Best REC: 0.9306994742509216
Best Params: {'C': 2.3167281368521384, 'solver': 'saga', 'max_iter': 935}
Other metrics: {'accuracy': 0.9701034879264085, 'recall': 0.9306994742509216, 'precision': 0.966573556131188, 'matthews_corrcoef': np.float64(0.9652643281758919), 'f1': 0.9455636755110491}


Ridge

In [ ]:
# ============================================================
# 3.2 RIDGE (L2) LOGISTIC REGRESSION - OPTUNA SEARCH
# ============================================================
# Search space: C in [1e-4, 10] (log), solver in {lbfgs, newton-cg, saga},
# max_iter in [100, 1000]; fixed penalty='l2', random_state=42.
# Objective: train on the noisy training set, predict the noisy test set,
# log accuracy / macro recall / macro precision / macro F1 / MCC, and
# return the macro recall. Optuna maximises it over 10 trials.
# (The cell re-imports the libraries it needs; this has no other effect.)
import optuna
import numpy as np
import cupy as cp
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, recall_score, precision_score, matthews_corrcoef

def objective(trial):
    # Hyperparameters for Ridge (L2) Logistic Regression
    params = {
        'C': trial.suggest_float('C', 0.0001, 10.0, log=True),   # inverse of regularization strength
        'solver': trial.suggest_categorical('solver', ['lbfgs', 'newton-cg', 'saga']),  # all support L2
        'max_iter': trial.suggest_int('max_iter', 100, 1000),
        'penalty': 'l2',  # Ridge regularization
        'random_state': 42
    }

    model = LogisticRegression(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    # Convert CuPy arrays to NumPy if using GPU arrays
    y_test_cpu = cp.asnumpy(y_test)
    preds_cpu = cp.asnumpy(preds)

    # Metrics
    f1 = f1_score(y_test_cpu, preds_cpu, average='macro')
    acc = accuracy_score(y_test_cpu, preds_cpu)
    rec = recall_score(y_test_cpu, preds_cpu, average='macro')
    preci = precision_score(y_test_cpu, preds_cpu, average='macro')
    matt = matthews_corrcoef(y_test_cpu, preds_cpu)

    # Log extra metrics
    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("recall", rec)
    trial.set_user_attr("precision", preci)
    trial.set_user_attr("matthews_corrcoef", matt)
    trial.set_user_attr("f1", f1)

    print(f"Trial {trial.number}: F1={f1:.4f}, Acc={acc:.4f}, Rec={rec:.4f}, Prec={preci:.4f}, MCC={matt:.4f}")

    return rec  # Optimize recall

# Run the optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# Display best results
best_trial = study.best_trial
print("\nBest REC:", best_trial.value)
print("Best Params:", best_trial.params)
print("Other metrics:", best_trial.user_attrs)

[I 2025-11-12 15:19:02,770] A new study created in memory with name: no-name-c1cc9a56-21a7-4c23-b726-871c0000292b
[I 2025-11-12 15:19:33,651] Trial 0 finished with value: 0.9254106832453273 and parameters: {'C': 6.96879662900301, 'solver': 'lbfgs', 'max_iter': 591}. Best is trial 0 with value: 0.9254106832453273.


Trial 0: F1=0.9413, Acc=0.9675, Rec=0.9254, Prec=0.9637, MCC=0.9622


[I 2025-11-12 15:19:40,956] Trial 1 finished with value: 0.8754776840710802 and parameters: {'C': 0.03257729893177855, 'solver': 'newton-cg', 'max_iter': 870}. Best is trial 0 with value: 0.9254106832453273.


Trial 1: F1=0.8977, Acc=0.9447, Rec=0.8755, Prec=0.9376, MCC=0.9358


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
[I 2025-11-12 15:19:48,441] Trial 2 finished with value: 0.702406422925742 and parameters: {'C': 0.0002879209983889449, 'solver': 'newton-cg', 'max_iter': 813}. Best is trial 0 with value: 0.9254106832453273.


Trial 2: F1=0.7151, Acc=0.8607, Rec=0.7024, Prec=0.7743, MCC=0.8394


... (10 outputs left out)

[I 2025-11-12 15:21:28,818] Trial 8 finished with value: 0.9253569321443329 and parameters: {'C': 5.574682826516398, 'solver': 'newton-cg', 'max_iter': 628}. Best is trial 3 with value: 0.9265969947050344.


Trial 8: F1=0.9416, Acc=0.9677, Rec=0.9254, Prec=0.9649, MCC=0.9625


[I 2025-11-12 15:24:42,557] Trial 9 finished with value: 0.9244749998634656 and parameters: {'C': 4.244985691469111, 'solver': 'saga', 'max_iter': 911}. Best is trial 3 with value: 0.9265969947050344.


Trial 9: F1=0.9410, Acc=0.9673, Rec=0.9245, Prec=0.9651, MCC=0.9621

Best REC: 0.9265969947050344
Best Params: {'C': 7.249509828331678, 'solver': 'newton-cg', 'max_iter': 786}
Other metrics: {'accuracy': 0.9681870448447681, 'recall': 0.9265969947050344, 'precision': 0.9651527361719283, 'matthews_corrcoef': np.float64(0.9630389093903181), 'f1': 0.9424682924606632}


# Elastic

In [ ]:
# ============================================================
# 3.3 ELASTIC-NET LOGISTIC REGRESSION - OPTUNA SEARCH
# ============================================================
# Search space: C in [1e-4, 10] (log), l1_ratio in [0, 1],
# max_iter in [100, 1000]; fixed penalty='elasticnet', solver='saga',
# random_state=42.
# Objective: train on the noisy training set, predict the noisy test set,
# log accuracy / macro recall / macro precision / macro F1 / MCC, and
# return the macro recall. Optuna maximises it over 10 trials.
def objective(trial):
    # Hyperparameters for Elastic Net Logistic Regression
    params = {
        'C': trial.suggest_float('C', 0.0001, 10.0, log=True),   # inverse of regularization strength
        'l1_ratio': trial.suggest_float('l1_ratio', 0.0, 1.0),   # balance between L1 (lasso) and L2 (ridge)
        'solver': 'saga',  # only saga supports elasticnet
        'penalty': 'elasticnet',
        'max_iter': trial.suggest_int('max_iter', 100, 1000),
        'random_state': 42
    }

    model = LogisticRegression(**params)
    model.fit(X_train, y_train)

    preds = model.predict(X_test)

    # Convert CuPy arrays to NumPy if using GPU arrays
    y_test_cpu = cp.asnumpy(y_test)
    preds_cpu = cp.asnumpy(preds)

    # Evaluation metrics
    f1 = f1_score(y_test_cpu, preds_cpu, average='macro')
    acc = accuracy_score(y_test_cpu, preds_cpu)
    rec = recall_score(y_test_cpu, preds_cpu, average='macro')
    preci = precision_score(y_test_cpu, preds_cpu, average='macro')
    matt = matthews_corrcoef(y_test_cpu, preds_cpu)

    # Log additional metrics
    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("recall", rec)
    trial.set_user_attr("precision", preci)
    trial.set_user_attr("matthews_corrcoef", matt)
    trial.set_user_attr("f1", f1)

    print(f"Trial {trial.number}: F1={f1:.4f}, Acc={acc:.4f}, Rec={rec:.4f}, Prec={preci:.4f}, MCC={matt:.4f}")

    return rec  # Optimize recall

# Run the optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# Display best results
best_trial = study.best_trial
print("\nBest REC:", best_trial.value)
print("Best Params:", best_trial.params)
print("Other metrics:", best_trial.user_attrs)

[I 2025-11-12 15:24:42,574] A new study created in memory with name: no-name-47679129-d678-4a62-9939-c4489f795b5f
[I 2025-11-12 15:25:28,560] Trial 0 finished with value: 0.8869961664377589 and parameters: {'C': 0.07055502039184407, 'l1_ratio': 0.7504746366613041, 'max_iter': 221}. Best is trial 0 with value: 0.8869961664377589.


Trial 0: F1=0.9080, Acc=0.9501, Rec=0.8870, Prec=0.9443, MCC=0.9421


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
[I 2025-11-12 15:25:40,924] Trial 1 finished with value: 0.5470431622197872 and parameters: {'C': 0.0006430316129389223, 'l1_ratio': 0.8690288236703939, 'max_iter': 757}. Best is trial 0 with value: 0.8869961664377589.


Trial 1: F1=0.5150, Acc=0.7472, Rec=0.5470, Prec=0.5200, MCC=0.7111


[I 2025-11-12 15:26:04,413] Trial 2 finished with value: 0.8777488274896141 and parameters: {'C': 0.040313264393156946, 'l1_ratio': 0.4795233884841036, 'max_iter': 251}. Best is trial 0 with value: 0.8869961664377589.


Trial 2: F1=0.8999, Acc=0.9462, Rec=0.8777, Prec=0.9408, MCC=0.9376


... (10 outputs left out)

[I 2025-11-12 15:30:08,921] Trial 8 finished with value: 0.9017035722782604 and parameters: {'C': 0.32421631749317886, 'l1_ratio': 0.26412068295144886, 'max_iter': 797}. Best is trial 8 with value: 0.9017035722782604.


Trial 8: F1=0.9211, Acc=0.9572, Rec=0.9017, Prec=0.9519, MCC=0.9504


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
[I 2025-11-12 15:30:23,028] Trial 9 finished with value: 0.5440737939853603 and parameters: {'C': 0.00019763985846916196, 'l1_ratio': 0.35041743084181154, 'max_iter': 308}. Best is trial 8 with value: 0.9017035722782604.


Trial 9: F1=0.5120, Acc=0.7435, Rec=0.5441, Prec=0.4975, MCC=0.7064

Best REC: 0.9017035722782604
Best Params: {'C': 0.32421631749317886, 'l1_ratio': 0.26412068295144886, 'max_iter': 797}
Other metrics: {'accuracy': 0.9572420254673992, 'recall': 0.9017035722782604, 'precision': 0.9518636137457477, 'matthews_corrcoef': np.float64(0.95037103378821), 'f1': 0.9210704403478367}


# svc

In [ ]:
# ============================================================
# 3.4 SUPPORT VECTOR MACHINE (cuML SVC) - OPTUNA SEARCH
# ============================================================
# Search space: C in [1e-3, 100] (log), kernel in {linear, rbf, poly, sigmoid},
# gamma in {scale, auto}, degree in [2, 5]; fixed probability=False,
# max_iter=1000.
# Objective: train on the noisy training set, predict the noisy test set,
# log accuracy / macro recall / macro precision / macro F1 / MCC, and
# return the macro recall. Optuna maximises it over 10 trials.
# Note: the final print is labelled 'Best F1' but shows the best macro recall.
# No output was saved for this cell.
def objective(trial):
    params = {
        'C': trial.suggest_float('C', 1e-3, 100.0, log=True),
        'kernel': trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid']),
        'gamma': trial.suggest_categorical('gamma', ['scale', 'auto']),
        'degree': trial.suggest_int('degree', 2, 5)
    }

    model = SVC(**params, probability=False, max_iter=1000)  # cuML GPU version
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # Convert predictions back to host for metrics
    preds_cpu = cp.asnumpy(preds)
    y_test_cpu = cp.asnumpy(y_test)

    f1 = f1_score(y_test_cpu, preds_cpu, average='macro')
    acc = accuracy_score(y_test_cpu, preds_cpu)
    rec = recall_score(y_test_cpu, preds_cpu, average='macro')
    preci = precision_score(y_test_cpu, preds_cpu, average='macro')
    matt = matthews_corrcoef(y_test_cpu, preds_cpu)

    # Log for inspection
    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("recall", rec)
    trial.set_user_attr("precision", preci)
    trial.set_user_attr("matthews_corrcoef", matt)
    trial.set_user_attr("f1", f1)

    print(f"Trial {trial.number}: F1={f1:.4f}, Acc={acc:.4f}, Rec={rec:.4f}, Prec={preci:.4f}, MCC={matt:.4f}")

    return rec

# Run Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

best_trial = study.best_trial
print("\nBest F1:", best_trial.value)
print("Best Params:", best_trial.params)
print("Other metrics:", best_trial.user_attrs)

# KNN

In [ ]:
# ============================================================
# 3.5 K-NEAREST NEIGHBORS - OPTUNA SEARCH
# ============================================================
# Search space: n_neighbors in [3, 25], weights in {uniform, distance},
# algorithm in {brute}, metric in {euclidean, manhattan, minkowski}, p in [1, 3].
# Note: KNeighborsClassifier here is the scikit-learn class imported above.
# Objective: train on the noisy training set, predict the noisy test set,
# log accuracy / macro recall / macro precision / macro F1 / MCC, and
# return the macro recall. Optuna maximises it over 10 trials.
# No output was saved for this cell.
def objective(trial):
    params = {
        'n_neighbors': trial.suggest_int('n_neighbors', 3, 25),
        'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
        'algorithm': trial.suggest_categorical('algorithm', ['brute']),  # cuML only supports 'brute'
        'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'minkowski']),
        'p': trial.suggest_int('p', 1, 3)  # only relevant if metric='minkowski'
    }

    model = KNeighborsClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # Convert to CPU for sklearn metrics
    preds_cpu = cp.asnumpy(preds)
    y_test_cpu = cp.asnumpy(y_test)

    f1 = f1_score(y_test_cpu, preds_cpu, average='macro')
    acc = accuracy_score(y_test_cpu, preds_cpu)
    rec = recall_score(y_test_cpu, preds_cpu, average='macro')
    preci = precision_score(y_test_cpu, preds_cpu, average='macro')
    matt = matthews_corrcoef(y_test_cpu, preds_cpu)

    # Log additional metrics
    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("recall", rec)
    trial.set_user_attr("precision", preci)
    trial.set_user_attr("matthews_corrcoef", matt)
    trial.set_user_attr("f1", f1)

    print(f"Trial {trial.number}: F1={f1:.4f}, Acc={acc:.4f}, Rec={rec:.4f}, Prec={preci:.4f}, MCC={matt:.4f}")

    return rec  # Optimize for REC

# Optimize with Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# Display best results
best_trial = study.best_trial
print("\Best REC:", best_trial.value)
print("Besnt Params:", best_trial.params)
print("Other metrics:", best_trial.user_attrs)

DECC

In [ ]:
# ============================================================
# 3.6 DECISION TREE - OPTUNA SEARCH
# ============================================================
# Search space: criterion in {gini, entropy}, max_depth in [3, 20],
# min_samples_split in [2, 10], min_samples_leaf in [1, 10] (no random_state).
# Objective: train on the noisy training set, predict the noisy test set,
# log accuracy / macro recall / macro precision / macro F1 / MCC, and
# return the macro recall. Optuna maximises it over 10 trials.
# No output was saved for this cell.
def objective(trial):
    params = {
        'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy']),
        'max_depth': trial.suggest_int('max_depth', 3, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
    }

    model = DecisionTreeClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # Convert to CPU for metrics
    preds_cpu = cp.asnumpy(preds)
    y_test_cpu = cp.asnumpy(y_test)

    # Metrics
    f1 = f1_score(y_test_cpu, preds_cpu, average='macro')
    acc = accuracy_score(y_test_cpu, preds_cpu)
    rec = recall_score(y_test_cpu, preds_cpu, average='macro')
    preci = precision_score(y_test_cpu, preds_cpu, average='macro')
    matt = matthews_corrcoef(y_test_cpu, preds_cpu)

    # Log to Optuna
    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("recall", rec)
    trial.set_user_attr("precision", preci)
    trial.set_user_attr("matthews_corrcoef", matt)
    trial.set_user_attr("f1", f1)

    print(f"Trial {trial.number}: F1={f1:.4f}, Acc={acc:.4f}, Rec={rec:.4f}, Prec={preci:.4f}, MCC={matt:.4f}")

    return rec
# Run Optuna study
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# Display best result
best_trial = study.best_trial
print("\nBest REC:", best_trial.value)
print("Best Params:", best_trial.params)
print("Other metrics:", best_trial.user_attrs)

# random

In [ ]:
# ============================================================
# 3.7 RANDOM FOREST - OPTUNA SEARCH
# ============================================================
# Search space: n_estimators in [100, 500], max_depth in [4, 20],
# min_samples_split in [2, 10], min_samples_leaf in [1, 10],
# bootstrap in {True, False} (no random_state).
# Objective: train on the noisy training set, predict the noisy test set,
# log accuracy / macro recall / macro precision / macro F1 / MCC, and
# return the macro recall. Optuna maximises it over 10 trials.
# Note: the first code line starts with one space (' def objective'),
# which raises an IndentationError when the cell is run as written.
# No output was saved for this cell.
 def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 4, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'bootstrap': trial.suggest_categorical('bootstrap', [True, False]),

    }

    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # Convert back to CPU for sklearn metrics
    preds_cpu = cp.asnumpy(preds)
    y_test_cpu = cp.asnumpy(y_test)

    # Compute metrics
    f1 = f1_score(y_test_cpu, preds_cpu, average='macro')
    acc = accuracy_score(y_test_cpu, preds_cpu)
    rec = recall_score(y_test_cpu, preds_cpu, average='macro')
    preci = precision_score(y_test_cpu, preds_cpu, average='macro')
    matt = matthews_corrcoef(y_test_cpu, preds_cpu)

    # Log metrics in Optuna
    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("recall", rec)
    trial.set_user_attr("precision", preci)
    trial.set_user_attr("matthews_corrcoef", matt)
    trial.set_user_attr("f1", f1)

    print(f"Trial {trial.number}: F1={f1:.4f}, Acc={acc:.4f}, Rec={rec:.4f}, Prec={preci:.4f}, MCC={matt:.4f}")

    return rec  # Optimize for rec score

# Run Optuna optimization
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# Best results
best_trial = study.best_trial
print("\nBest rec:", best_trial.value)
print("Best Params:", best_trial.params)
print("Other metrics:", best_trial.user_attrs)

### Logistic Regression

In [ ]:
# ============================================================
# 3.8 LOGISTIC REGRESSION - OPTUNA SEARCH
# ============================================================
# Search space: C in [1e-3, 100] (log), penalty in {l2},
# fit_intercept in {True, False}, max_iter in [100, 1000],
# tol in [1e-6, 1e-3] (log); default lbfgs solver, no random_state.
# Objective: train on the noisy training set, predict the noisy test set,
# log accuracy / macro recall / macro precision / macro F1 / MCC, and
# return the macro recall. Optuna maximises it over 10 trials.
# No output was saved for this cell.
def objective(trial):
    params = {
        'C': trial.suggest_float('C', 1e-3, 100.0, log=True),
        'penalty': trial.suggest_categorical('penalty', ['l2']),
        'fit_intercept': trial.suggest_categorical('fit_intercept', [True, False]),
        'max_iter': trial.suggest_int('max_iter', 100, 1000),
        'tol': trial.suggest_float('tol', 1e-6, 1e-3, log=True),
    }

    model = LogisticRegression(**params)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    # Move back to CPU for metrics
    preds_cpu = cp.asnumpy(preds)
    y_test_cpu = cp.asnumpy(y_test)

    # Compute metrics
    f1 = f1_score(y_test_cpu, preds_cpu, average='macro')
    acc = accuracy_score(y_test_cpu, preds_cpu)
    rec = recall_score(y_test_cpu, preds_cpu, average='macro')
    preci = precision_score(y_test_cpu, preds_cpu, average='macro')
    matt = matthews_corrcoef(y_test_cpu, preds_cpu)

    # Log metrics for inspection
    trial.set_user_attr("accuracy", acc)
    trial.set_user_attr("recall", rec)
    trial.set_user_attr("precision", preci)
    trial.set_user_attr("matthews_corrcoef", matt)
    trial.set_user_attr("f1", f1)

    print(f"Trial {trial.number}: F1={f1:.4f}, Acc={acc:.4f}, Rec={rec:.4f}, Prec={preci:.4f}, MCC={matt:.4f}")

    return rec  # Optimize F1 score

# Run Optuna
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=10)

# Show best result
best_trial = study.best_trial
print("\nBest rec", best_trial.value)
print("Best Params:", best_trial.params)
print("Other metrics:", best_trial.user_attrs)

## 4. Draft ROC code (exploratory, not functional)

The last cells are an unfinished draft of the ROC plotting code. They are tested on **random placeholder data**: `np.random.randint` predictions and a random `y_test`. Cell 31 also **overwrites `y_test`** with random values. As written, the draft would not run to completion:

* `results[:,0]` indexes a dictionary, which raises a `TypeError`;
* the plotting cell uses `plt`, `roc_auc_score` and `y_score`, which are not imported or defined in this notebook.

These cells have no saved outputs and do not contribute to any reported result. The working ROC code is in notebooks `02`, `03` and `04b`.

In [ ]:
# ============================================================
# 4. DRAFT ROC CODE (EXPLORATORY, RANDOM PLACEHOLDER DATA)
# ============================================================
# Import the random module (not used below).
import random

In [ ]:
# Random placeholder 'predictions' (3 samples) used only to test the draft plotting code.
pre_xgb = np.random.randint(0, 9, (3, 1))
pre_svc=np.random.randint(0, 9, (3, 1))
pre_knn=np.random.randint(0, 9, (3, 1))
pre_dtree=np.random.randint(0, 9, (3, 1))
pre_lg=np.random.randint(0, 9, (3, 1))
pre_rf=np.random.randint(0, 9, (3, 1))

In [ ]:
# Random placeholder labels: this overwrites the real y_test from the split.
y_test = np.random.randint(0, 9, (3, 1))

In [ ]:
# Dictionary of placeholder predictions per model.
results = {

    "XGBoost":  pre_xgb,
    "Support Vector Machine": pre_svc,
    "k-Nearest Neighbors": pre_knn,
    "decision tree": pre_dtree,
    "Logistic Regression": pre_lg,
    "Random Forest": pre_rf

}

In [ ]:
# Draft check: indexing a dictionary like an array raises a TypeError.
results[:,0]

In [ ]:
# Initialise per-class containers for the ROC computation.
n_classes = 10
fpr = [0] * n_classes
tpr = [0] * n_classes
thresholds = [0] * n_classes
auc_score = [0] * n_classes

In [ ]:
# Draft one-vs-rest ROC plot. As written it relies on `plt`, `roc_auc_score`
# and `y_score`, which are not imported or defined in this notebook.
from sklearn.preprocessing import label_binarize
from sklearn.multiclass import OneVsRestClassifier
from sklearn.metrics import roc_curve, auc
results = {

    "XGBoost":  pre_xgb,
    "Support Vector Machine": pre_svc,
    "k-Nearest Neighbors": pre_knn,
    "decision tree": pre_dtree,
    "Logistic Regression": pre_lg,
    "Random Forest": pre_rf

}

# "Random Forest": y_pred_rf, et "k-Nearest Neighbors": y_pred_knn ajout apres
n_classes = 10
fpr = [0] * n_classes
print(fpr)
tpr = [0] * n_classes
print(tpr)
thresholds = [0] * n_classes
auc_score = [0] * n_classes





plt.figure()

for name, y in results.items():

    for i in range(n_classes):
        fpr[i], tpr[i], thresholds[i] = roc_curve(y_test[:, i],y[:, i])
        auc_score[i] = auc(fpr[i], tpr[i])

    # plt.plot(fpr, tpr, lw=2, label='%s (area = %0.2f)' % (name, auc_score))
    # #plt.fill_between(fpr, tpr, alpha=0.4)
    # Affichage
    auc_score["macro"] = roc_auc_score(y_test, y_score, average="macro")
    for i in range(n_classes):
        plt.plot(fpr[i], tpr[i], label=f"Classe {i} (AUC = {auc_score[i]:.2f})")


# Plot the diagonal line representing random guessing
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')

# Customize the plot
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curves')
plt.legend(loc='lower right')

plt.savefig("ROC_NONOISE_NOHYPER.png", dpi=300, bbox_inches='tight')
# Show the plot
plt.show()